# Volcano Seismo-Acoustic Detection — Portfolio Pipeline
Compact, runnable reconstruction of the thesis-style workflow using synthetic coupled events. Real MiniSEED and institutional catalogs are intentionally not redistributed.


In [ ]:
import numpy as np
from src.filters import filter_component
from src.features import window_features
from src.clustering import kmeans_candidates
from src.validation import validate_coupling
from src.spectra import amplitude_spectrum


In [ ]:
fs = 50.0
seconds = 180
t = np.arange(int(fs*seconds))/fs
rng = np.random.default_rng(42)
acoustic = 0.08*rng.normal(size=len(t))
seismic = 0.08*rng.normal(size=len(t))
for center in [35, 92, 145]:
    env = np.exp(-0.5*((t-center)/1.8)**2)
    acoustic += 1.3*env*np.sin(2*np.pi*2.2*t)
    seismic += 1.0*env*np.sin(2*np.pi*3.0*t) + 0.45*env*np.sin(2*np.pi*11*t)


In [ ]:
isa = filter_component(acoustic, fs, 'ISA')
smp = filter_component(seismic, fs, 'SMP')
ist = filter_component(seismic, fs, 'IST')
features = window_features(smp, fs)
clustered, model, high_cluster = kmeans_candidates(features)
clustered[clustered.candidate].head()


In [ ]:
center = 92
half = int(3*fs)
i = int(center*fs)
validate_coupling(isa[i-half:i+half], smp[i-half:i+half], fs)


## Real-data workflow
The full research workflow adds MiniSEED ingestion with ObsPy, event merging within short time gaps, statistics such as kurtosis/skewness/KDE, multi-component validation, and comparison against the INSIVUMEH catalog.
